# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VenkataVishnuVardhanReddy/Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

We code the feature engineering vector here:
1. Impute missing numeric variables (e.g., search volume, competition, word count) with their median value.
2. Label encode the categorical metadata fields (`content_type`, `main_intent`) to represent them numerically.
3. Construct the clean feature matrix `X` and the binary target vector `y` (pages experiencing a downward trend).

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
y = df['trend_direction'].str.lower().eq('down').astype(int)

# 1. Impute missing values
df['search_volume'] = df['search_volume'].fillna(df['search_volume'].median())
df['competition'] = df['competition'].fillna(df['competition'].median())
df['word_count'] = df['word_count'].fillna(df['word_count'].median())
df['char_count'] = df['char_count'].fillna(df['char_count'].median())
df['scroll_rate'] = df['scroll_rate'].fillna(df['scroll_rate'].median())

# 2. Encode categoricals
le = LabelEncoder()
df['content_type_enc'] = le.fit_transform(df['content_type'].astype(str))
df['main_intent_enc'] = le.fit_transform(df['main_intent'].astype(str))

# 3. Features
features = [
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count', 'char_count',
    'search_volume', 'competition', 'content_type_enc', 'main_intent_enc'
]

X = df[features]
print(f"Feature matrix X constructed with shape: {X.shape}")
print(f"Target vector y constructed with shape:  {y.shape}")


Feature matrix X constructed with shape: (30000, 22)
Target vector y constructed with shape:  (30000,)


## 2. Feature notes (meaning, missing, categorical, available-when?)

For our feature set, we confirm their availability at the prediction moment and document their profiles:

| Feature Name | Type | Imputation Strategy | Available Before Prediction? | Meaning |
| :--- | :--- | :--- | :--- | :--- |
| `impressions_90d` | Numeric | None (complete) | Yes | Total page impressions in GSC over trailing 90 days. |
| `clicks_90d` | Numeric | None (complete) | Yes | Total page clicks in GSC over trailing 90 days. |
| `ctr` | Numeric | None (complete) | Yes | Click-through rate (clicks / impressions) over 90 days. |
| `avg_position` | Numeric | None (complete) | Yes | Average ranking position of the page in search results. |
| `content_age_days` | Numeric | None (complete) | Yes | Days since the page content was originally published. |
| `days_since_last_update` | Numeric | None (complete) | Yes | Days since the last recorded refresh review. |
| `scroll_rate` | Numeric | Median | Yes | Percentage of sessions with page scroll events (GA4). |
| `word_count` | Numeric | Median | Yes | Total number of words in the article content. |
| `search_volume` | Numeric | Median | Yes | Estimated monthly search volume of the target keyword. |
| `content_type_enc` | Categorical | None | Yes | Label-encoded content structure category (e.g. blog, landing). |
| `main_intent_enc` | Categorical | None | Yes | Label-encoded search intent classification. |

In [2]:
# Output target features profile for documentation
feature_info = pd.DataFrame({
    'Feature': X.columns,
    'Dtype': X.dtypes.values,
    'Null_Count': X.isnull().sum().values
})
print("Feature profiling summary:")
print(feature_info.to_string(index=False))


Feature profiling summary:
               Feature   Dtype  Null_Count
       impressions_90d   int64           0
            clicks_90d   int64           0
         pageviews_90d   int64           0
          sessions_90d   int64           0
             users_90d   int64           0
  engaged_sessions_90d   int64           0
     scroll_events_90d   int64           0
 days_with_impressions   int64           0
    days_with_sessions   int64           0
      content_age_days   int64           0
days_since_last_update   int64           0
                   ctr float64           0
          avg_position float64           0
       engagement_rate float64           0
           scroll_rate float64           0
        ai_traffic_pct float64           0
            word_count float64           0
            char_count float64           0
         search_volume float64           0
           competition float64           0
      content_type_enc   int32           0
       main_intent_enc   in

## 3. The leakage hunt

We perform a leakage test by comparing two model configurations evaluated under **GroupKFold cross-validation (grouped by `client_id`)**:
- **Set A (Leaky):** Includes the 30-day sub-window metrics (`impressions_last_30d`, `impressions_prev_30d`), which partition the outcome window that determines the decline label.
- **Set B (Honest):** Uses only clean trailing 90-day features.

We demonstrate that Set A leaks outcome information and inflates out-of-fold performance:

In [3]:
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Pre-load comparison features
leaky_feats = features + ['impressions_last_30d', 'impressions_prev_30d']
honest_feats = features

gkf = GroupKFold(n_splits=3)
groups = df['client_id']

for name, feats in [("Leaky Configuration (Set A)", leaky_feats), ("Honest Configuration (Set B)", honest_feats)]:
    aucs = []
    # Test out-of-fold GroupKFold validation
    for train_idx, val_idx in gkf.split(df, y, groups):
        train_X, val_X = df[feats].iloc[train_idx], df[feats].iloc[val_idx]
        train_y, val_y = y.iloc[train_idx], y.iloc[val_idx]
        
        clf = RandomForestClassifier(n_estimators=30, max_depth=6, random_state=42, n_jobs=-1)
        clf.fit(train_X, train_y)
        preds = clf.predict_proba(val_X)[:, 1]
        aucs.append(roc_auc_score(val_y, preds))
    
    print(f"{name} Out-of-Fold Average ROC-AUC: {np.mean(aucs):.4f}")


Leaky Configuration (Set A) Out-of-Fold Average ROC-AUC: 0.7954


Honest Configuration (Set B) Out-of-Fold Average ROC-AUC: 0.6739


## 4. What I excluded and why

We explicitly exclude the following fields from model training to prevent leakage and bias:

1. **`impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d`**:
   - *Reason:* Direct label leakage. These sub-window metrics are what the target trend is calculated from, causing the model to learn mathematical relationships in the outcome window rather than search performance signals.
2. **`trend_pct`**:
   - *Reason:* Sibling to the target label. Contains future rate of change details.
3. **`provider_used`, `model_used`**:
   - *Reason:* Highly sparse (up to 71.5% missing) and represents internal workflow decisions (which writer or LLM model created the text), not search metrics. Including them introduces severe dataset imputation bias.

In [4]:
# Output the list of excluded fields for verification
excluded_list = [
    'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d',
    'sessions_last_30d', 'sessions_prev_30d', 'provider_used', 'model_used', 'trend_pct'
]
print("Verified Excluded Fields:")
for col in excluded_list:
    print(f" - {col}")


Verified Excluded Fields:
 - impressions_last_30d
 - impressions_prev_30d
 - clicks_last_30d
 - clicks_prev_30d
 - sessions_last_30d
 - sessions_prev_30d
 - provider_used
 - model_used
 - trend_pct


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.